# WNBA Knowledge Graph 

Dataset: 763 pemain WNBA, 20 tim, 143 universitas, 9 posisi (`dataset_WNBA.csv`).


In [1]:
import sys
sys.path.append("../src")
import pandas as pd


## Dataset

In [2]:
df = pd.read_csv("/Users/byg/Desktop/GRAF UAS WNBA/WNBA-Gaph-FinalProject/data/dataset_WNBA.csv")
print(df.shape)
df.head()


(1328, 6)


,playerLabel,teamLabel,collegeLabel,positionLabel,height,draftYear
0,Megan Gustafson,Las Vegas Aces,University of Iowa,center,75.0,2019.0
1,Awak Kuier,Dallas Wings,NaN,power forward,195.0,2021.0
2,Aliyah Boston,Indiana Fever,University of South Carolina,NaN,196.0,2023.0
3,Jackie Young,Las Vegas Aces,University of Notre Dame,shooting guard,183.0,2019.0
4,Sabrina Ionescu,New York Liberty,University of Oregon,point guard,180.0,2020.0


In [3]:
print("Jumlah pemain unik :", df['playerLabel'].nunique())
print("Jumlah tim          :", df['teamLabel'].nunique())
print("Jumlah universitas  :", df['collegeLabel'].nunique())
print("Posisi              :", df['positionLabel'].dropna().unique())


Jumlah pemain unik : 763
Jumlah tim          : 20
Jumlah universitas  : 143
Posisi              : ['center' 'power forward' 'shooting guard' 'point guard' 'small forward'
 'forward' 'guard' 'combo guard']


## 1. Koneksi Database 

Membuktikan koneksi ke Neo4j berhasil dan menampilkan ringkasan jumlah node per label.


In [4]:
from db import get_connection

conn = get_connection()
print("Koneksi berhasil:", conn.verify_connectivity())

summary = conn.run(
    "MATCH (n) RETURN labels(n)[0] AS label, count(*) AS jumlah ORDER BY jumlah DESC"
)
pd.DataFrame(summary)


Koneksi berhasil: True


,label,jumlah
0,Player,763
1,College,143
2,Team,20
3,Position,8


## 2. Skema Graph

```
(:Player {name, height, draftYear})
(:Team {name})
(:College {name})
(:Position {name})

(Player)-[:PLAYS_FOR]->(Team)
(Player)-[:ATTENDED]->(College)
(Player)-[:PLAYS_POSITION]->(Position)
```

4 jenis entitas (>3 entitas berbeda ✅) dan >50 node dengan relasi bermakna ✅.


## 3. Graph Analytics 



In [5]:
# Contoh: Degree & Betweenness centrality pada Player (GDS)
# -> Sesuaikan dengan query centrality yang SUDAH kamu jalankan sebelumnya.

centrality_query = '''
CALL gds.graph.exists('wnbaGraph') YIELD exists
RETURN exists
'''
print(conn.run(centrality_query))


[{'exists': False}]


In [6]:
# Contoh traversal sederhana: pemain & tim yang terhubung lewat universitas yang sama
traversal_query = '''
MATCH (p1:Player)-[:EDUCATED_AT]->(c:College)<-[:EDUCATED_AT]-(p2:Player)
WHERE p1.name < p2.name
RETURN c.name AS college, p1.name AS player1, p2.name AS player2
LIMIT 10
'''
pd.DataFrame(conn.run(traversal_query))

,college,player1,player2
0,University of Iowa,Kate Martin,Megan Gustafson
1,University of Iowa,Caitlin Clark,Megan Gustafson
2,University of Iowa,Caitlin Clark,Kate Martin
3,University of South Carolina,A'ja Wilson,Aliyah Boston
4,University of South Carolina,Aliyah Boston,Shannon Johnson
5,University of South Carolina,A'ja Wilson,Shannon Johnson
6,University of South Carolina,Brian Winters,Shannon Johnson
7,University of South Carolina,Jocelyn Penn,Shannon Johnson
8,University of South Carolina,Allisha Gray,Shannon Johnson
9,University of South Carolina,Mikiah Herbert Harrigan,Shannon Johnson


## 4. LLM untuk Text-to-Cypher 

In [7]:
from text_to_cypher import ask

hasil = ask("Sebutkan 5 pemain dengan tinggi badan tertinggi beserta universitasnya")
print("Cypher yang dihasilkan Gemini:\n", hasil["cypher"])
print()
print("Jawaban:", hasil["answer"])
pd.DataFrame(hasil["rows"])


Cypher yang dihasilkan Gemini:
 MATCH (p:Player)-[:EDUCATED_AT]->(c:College)
WHERE p.height IS NOT NULL
WITH p, c, p.height AS tinggi
ORDER BY tinggi DESC
RETURN p.name AS Nama, c.name AS Universitas, tinggi AS Tinggi
LIMIT 5

Jawaban: Berikut adalah 5 pemain dengan tinggi badan tertinggi beserta universitasnya:

1. Tree Rollins dari Clemson University dengan tinggi 216 cm
2. Orlando Woolridge dari University of Notre Dame dengan tinggi 206 cm
3. Brittney Griner dari Baylor University dengan tinggi 203 cm
4. Katie Feenstra-Mattera dari Liberty University dengan tinggi 203 cm
5. Tidak ada informasi tentang pemain lain yang lebih tinggi dari mereka.


,Nama,Universitas,Tinggi
0,Tree Rollins,Clemson University,216.0
1,Orlando Woolridge,University of Notre Dame,206.0
2,Brittney Griner,Baylor University,203.0
3,Katie Feenstra-Mattera,Liberty University,203.0
4,Alison Bales,Duke University,201.0


In [8]:
hasil2 = ask("Universitas mana yang paling banyak menghasilkan pemain WNBA?")
print(hasil2["cypher"])
print(hasil2["answer"])
pd.DataFrame(hasil2["rows"])


Received notification from DBMS server: {severity: WARNING} {code: Neo.ClientNotification.Statement.UnknownPropertyKeyWarning} {category: UNRECOGNIZED} {title: The provided property key is not in the database} {description: One of the property names in your query is not available in the database, make sure you didn't misspell it or that the label is available when you run this statement in your application (the missing property name is: collegeName)} {position: line: 1, column: 137, offset: 136} for query: "MATCH (c:College)-[:EDUCATED_AT]->(p:Player)-[:PLAYS_FOR]->(t:Team)<-[:PLAYS_FOR]-(p2:Player) WHERE p2.position = 'WNBA' AND c.name = p.collegeName RETURN c.name AS Universitas, COUNT(p2) AS Jumlah_Pemain"


MATCH (c:College)-[:EDUCATED_AT]->(p:Player)-[:PLAYS_FOR]->(t:Team)<-[:PLAYS_FOR]-(p2:Player) WHERE p2.position = 'WNBA' AND c.name = p.collegeName RETURN c.name AS Universitas, COUNT(p2) AS Jumlah_Pemain
Mohon maaf, saya tidak dapat menemukan informasi tentang universitas yang paling banyak menghasilkan pemain WNBA.


""


## 5. Machine Learning pada Graph (GDS)

FastRP node embedding → K-Means clustering pemain → klasifikasi posisi pemain
menggunakan embedding + tinggi badan.


In [9]:
from graph_ml import get_gds, project_graph, run_node_embedding, run_clustering, run_position_classification

gds = get_gds()
graph = project_graph(gds)
print(graph.node_count(), "node |", graph.relationship_count(), "relasi di-projeksikan")


/Users/byg/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


934 node | 4736 relasi di-projeksikan


In [10]:
run_node_embedding(gds, graph)
clusters = run_clustering(gds, n_clusters=5)
clusters.head(10)


,name,cluster
0,Megan Gustafson,3
1,Awak Kuier,0
2,Aliyah Boston,0
3,Jackie Young,3
4,Sabrina Ionescu,2
5,Lou Lopez Sénéchal,2
6,Kate Martin,3
7,Cameron Brink,2
8,Rickea Jackson,2
9,Rhyne Howard,2


In [11]:
report = run_position_classification(gds)
print(report)


                precision    recall  f1-score   support

        center       0.90      0.93      0.91        40
       forward       0.00      0.00      0.00         1
   point guard       0.88      0.92      0.90        25
 power forward       0.84      0.62      0.71        26
shooting guard       0.90      0.90      0.90        42
 small forward       0.77      0.93      0.84        29

      accuracy                           0.87       163
     macro avg       0.72      0.72      0.71       163
  weighted avg       0.86      0.87      0.86       163



In [12]:
graph.drop()
gds.close()


## 6. LLM for Graph Builder 

Ekstraksi entitas & relasi dari teks bebas (misal cuplikan berita/bio pemain),
lalu populate otomatis ke Neo4j. (Screenshot B)


In [13]:
from graph_builder import build_graph_from_text

teks_bebas = '''
A'ja Wilson bermain untuk Las Vegas Aces dan merupakan alumni dari
University of South Carolina. Pada tahun 2024, ia memenangkan WNBA
Most Valuable Player Award untuk ketiga kalinya dan membawa Las Vegas
Aces tampil di WNBA Finals.
'''

hasil_builder = build_graph_from_text(teks_bebas)
print(hasil_builder["extracted"])
print("Ditulis ke Neo4j:", hasil_builder["write_summary"])


{'entities': [{'type': 'Player', 'name': "A'ja Wilson"}, {'type': 'Team', 'name': 'Las Vegas Aces'}, {'type': 'College', 'name': 'University of South Carolina'}, {'type': 'Award', 'name': 'WNBA Most Valuable Player Award'}, {'type': 'Event', 'name': 'WNBA Finals'}], 'relations': [{'source': "A'ja Wilson", 'type': 'PLAYS_FOR', 'target': 'Las Vegas Aces'}, {'source': "A'ja Wilson", 'type': 'ATTENDED', 'target': 'University of South Carolina'}, {'source': "A'ja Wilson", 'type': 'WON', 'target': 'WNBA Most Valuable Player Award'}, {'source': 'Las Vegas Aces', 'type': 'PARTICIPATED_IN', 'target': 'WNBA Finals'}]}
Ditulis ke Neo4j: {'entities_written': 5, 'relations_written': 4}


## 7. RAG (Graph-Augmented Retrieval) 

Mengambil subgraph relevan sebagai konteks, lalu menjawab pertanyaan dengan Gemini
secara grounded pada data graph (mengurangi halusinasi). (Screenshot D)


In [14]:
from rag import rag_answer

res = rag_answer("Apa hubungan antara A'ja Wilson dan Las Vegas Aces?")
print("Konteks subgraph:\n", res["context"])
print()
print("Jawaban RAG:", res["answer"])


Konteks subgraph:
 Las Vegas Aces -[PLAYS_FOR]- Megan Gustafson (Player)
Las Vegas Aces -[PLAYS_FOR -> PLAYS_FOR]- Dallas Wings (Team)
Las Vegas Aces -[PLAYS_FOR -> PLAYS_FOR]- Washington Mystics (Team)
Las Vegas Aces -[PLAYS_FOR -> PLAYS_FOR]- Phoenix Mercury (Team)
Las Vegas Aces -[PLAYS_FOR -> EDUCATED_AT]- University of Iowa (College)
Las Vegas Aces -[PLAYS_FOR -> PLAYS_POSITION]- center (Position)
Las Vegas Aces -[PLAYS_FOR -> ATTENDED]- University of Iowa (College)
Las Vegas Aces -[PLAYS_FOR]- Jackie Young (Player)
Las Vegas Aces -[PLAYS_FOR -> EDUCATED_AT]- University of Notre Dame (College)
Las Vegas Aces -[PLAYS_FOR -> PLAYS_POSITION]- shooting guard (Position)
Las Vegas Aces -[PLAYS_FOR -> ATTENDED]- University of Notre Dame (College)
Las Vegas Aces -[PLAYS_FOR]- Kate Martin (Player)
Las Vegas Aces -[PLAYS_FOR]- A'ja Wilson (Player)
Las Vegas Aces -[PLAYS_FOR -> EDUCATED_AT]- University of South Carolina (College)
Las Vegas Aces -[PLAYS_FOR -> ATTENDED]- University of South C

In [15]:
res2 = rag_answer("Pemain mana saja dari University of Iowa yang ada di graph ini?")
print(res2["context"])
print()
print(res2["answer"])


University of Notre Dame -[EDUCATED_AT]- Jackie Young (Player)
University of Notre Dame -[EDUCATED_AT -> PLAYS_FOR]- Las Vegas Aces (Team)
University of Notre Dame -[EDUCATED_AT -> PLAYS_POSITION]- shooting guard (Position)
University of Notre Dame -[EDUCATED_AT -> ATTENDED]- University of Notre Dame (College)
University of Notre Dame -[EDUCATED_AT]- Arike Ogunbowale (Player)
University of Notre Dame -[EDUCATED_AT -> PLAYS_FOR]- Dallas Wings (Team)
University of Notre Dame -[EDUCATED_AT -> PLAYS_POSITION]- point guard (Position)
University of Notre Dame -[EDUCATED_AT]- Devereaux Peters (Player)
University of Notre Dame -[EDUCATED_AT -> PLAYS_FOR]- Minnesota Lynx (Team)
University of Notre Dame -[EDUCATED_AT -> PLAYS_POSITION]- small forward (Position)
University of Notre Dame -[EDUCATED_AT]- Ruth Riley (Player)
University of Notre Dame -[EDUCATED_AT -> PLAYS_FOR]- Atlanta Dream (Team)
University of Notre Dame -[EDUCATED_AT -> PLAYS_FOR]- Chicago Sky (Team)
University of Notre Dame -[ED

In [ ]:
# Close Connection
conn.close()
print("Selesai.")


Selesai.
